In [ ]:
from wav2vec2_vib import Model 

model = Model()

model.load_state_dict(torch.load('KDW2V-AASISTL_new/Distil-W2V2-AASIST.pth'))

# TIMIT_TTS

In [ ]:
import os

# Define the paths to the files (modify these paths as needed)
protocol_file_path = "/data/Datasets/Latin_America_Spanish_anti_spoofing_dataset/FinalDataset_16khz/protocol.txt"
tree_file_path = "/data/Datasets/Latin_America_Spanish_anti_spoofing_dataset/FinalDataset_16khz/tree.txt"
output_file_path = "/data/Datasets/Latin_America_Spanish_anti_spoofing_dataset/FinalDataset_16khz/protocol_updated.txt"

# Helper function to read file with different encodings
def read_file(file_path):
    for encoding in ('utf-8', 'latin-1', 'iso-8859-1'):
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.readlines()
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Unable to decode the file {file_path} with known encodings.")

# Read the protocol.txt file
protocol_lines = read_file(protocol_file_path)

# Read the tree.txt file
tree_lines = read_file(tree_file_path)

# Create a mapping from original name to destination name
name_mapping = {}
for line in tree_lines:
    if '.wav' in line:
        parts = line.strip().split('/')
        if len(parts) >= 2:
            original_name = parts[-1].replace('.wav', '')
            destination_name = parts[-2]
            name_mapping[original_name] = destination_name
        else:
            print(f"Skipping line due to unexpected format: {line}")

# Rewrite the protocol.txt with updated names
with open(output_file_path, 'w') as output_file:
    for line in protocol_lines:
        parts = line.strip().split()
        original_name = parts[1]
        if original_name in name_mapping:
            parts[1] = name_mapping[original_name]
        output_file.write(' '.join(parts) + '\n')

print("protocol.txt has been successfully updated!")


In [1]:
from wav2vec2_vib import Model as Wav2Vec2VIB
from student import Distil_XLSR_N_Trans_Layer_VIB
import torch
from torchinfo import summary
from main import  set_random_seed
import numpy as np
# Set the random seed for numpy and PyTorch
np.random.seed(42)
#set_random_seed(42)

# Ensure deterministic behavior in cudnn
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



device, ssl_cpkt_path = "cuda", "/datab/hungdx/KDW2V-AASISTL/xlsr2_300m.pt"
model = Distil_XLSR_N_Trans_Layer_VIB(device, ssl_cpkt_path).to(device)

input_size = torch.randn(1, 16000).to(device)

model.eval()

with torch.no_grad():
    res1 = model(input_size)
    print(res1)
    res2 = model(input_size)
    print(res2)
    res3 = model(input_size)
    print(res3)

#model.ssl_model.model.cfg.encoder_layerdrop = 0.5

# print("After layerdrop")
# with torch.no_grad():
#     res2 = model(input_size)
# # check if the output is the same
#     print(res2)


#summary(model, input_size=(1,  16000), depth=5)
# Laod pre-trained model
# model.load_state_dict(torch.load(
#     "/datad/hungdx/KDW2V-AASISTL/pretrained/vib_conf-5_gelu_acmccs_apr3_moreko_epoch22.pth"))

/home/hungdx/miniconda3/envs/KDW2V2-AASISTL/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


tensor([[ 0.1172, -0.0249]], device='cuda:0')
tensor([[ 0.1123, -0.0245]], device='cuda:0')
tensor([[ 0.1157, -0.0225]], device='cuda:0')


In [4]:
model

Distil_XLSR_N_Trans_Layer_VIB(
  (ssl_model): My_XLSR_FE(
    (model): Wav2Vec2Model(
      (feature_extractor): ConvFeatureExtractionModel(
        (conv_layers): ModuleList(
          (0): Sequential(
            (0): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
            (1): Dropout(p=0.0, inplace=False)
            (2): Sequential(
              (0): TransposeLast()
              (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (2): TransposeLast()
            )
            (3): GELU(approximate='none')
          )
          (1-4): 4 x Sequential(
            (0): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
            (1): Dropout(p=0.0, inplace=False)
            (2): Sequential(
              (0): TransposeLast()
              (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (2): TransposeLast()
            )
            (3): GELU(approximate='none')
          )
          (5-6): 2 x Sequential(
            (0): Con

In [12]:
model.named_modules

<bound method Module.named_modules of Distil_XLSR_N_Trans_Layer_VIB(
  (ssl_model): My_XLSR_FE(
    (model): Wav2Vec2Model(
      (feature_extractor): ConvFeatureExtractionModel(
        (conv_layers): ModuleList(
          (0): Sequential(
            (0): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
            (1): Dropout(p=0.0, inplace=False)
            (2): Sequential(
              (0): TransposeLast()
              (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (2): TransposeLast()
            )
            (3): GELU(approximate='none')
          )
          (1-4): 4 x Sequential(
            (0): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
            (1): Dropout(p=0.0, inplace=False)
            (2): Sequential(
              (0): TransposeLast()
              (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (2): TransposeLast()
            )
            (3): GELU(approximate='none')
          )
          (5-6)

In [13]:
for name, param in model.named_parameters():
    if 'ssl_model.model.feature_extractor' in name:
        param.requires_grad = False

summary(model, input_size=(1,  16000))


Layer (type:depth-idx)                                            Output Shape              Param #
Distil_XLSR_N_Trans_Layer_VIB                                     [1, 2]                    --
├─My_XLSR_FE: 1-1                                                 [1, 49, 1024]             --
│    └─Wav2Vec2Model: 2-1                                         [49, 1, 1024]             1,952,896
│    │    └─ConvFeatureExtractionModel: 3-1                       [1, 512, 49]              (4,210,176)
│    │    └─LayerNorm: 3-2                                        [1, 49, 512]              1,024
│    │    └─Linear: 3-3                                           [1, 49, 1024]             525,312
│    │    └─Dropout: 3-4                                          [1, 49, 1024]             --
│    │    └─Dropout: 3-5                                          [1, 49, 512]              --
│    │    └─TransformerEncoder: 3-6                               [1, 49, 1024]             71,372,928
├─Linear: 1-2

In [14]:
summary(model, input_size=(1,  16000))

Layer (type:depth-idx)                                            Output Shape              Param #
Distil_XLSR_N_Trans_Layer_VIB                                     [1, 2]                    --
├─My_XLSR_FE: 1-1                                                 [1, 49, 1024]             --
│    └─Wav2Vec2Model: 2-1                                         [49, 1, 1024]             1,952,896
│    │    └─ConvFeatureExtractionModel: 3-1                       [1, 512, 49]              (4,210,176)
│    │    └─LayerNorm: 3-2                                        [1, 49, 512]              1,024
│    │    └─Linear: 3-3                                           [1, 49, 1024]             525,312
│    │    └─Dropout: 3-4                                          [1, 49, 1024]             --
│    │    └─Dropout: 3-5                                          [1, 49, 512]              --
│    │    └─TransformerEncoder: 3-6                               [1, 49, 1024]             71,372,928
├─Linear: 1-2

In [15]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    target_modules=["q_proj", "v_proj", "k_proj"],    
)
peft_model = get_peft_model(model, config)

print(peft_model(torch.randn(1, 16000).to(device)).shape)
#summary(peft_model, input_size=(1,  16000))

torch.Size([1, 2])


In [16]:
peft_model.save_pretrained("/datad/hungdx/KDW2V-AASISTL/peft_model")

In [17]:
model

Distil_XLSR_N_Trans_Layer_VIB(
  (ssl_model): My_XLSR_FE(
    (model): Wav2Vec2Model(
      (feature_extractor): ConvFeatureExtractionModel(
        (conv_layers): ModuleList(
          (0): Sequential(
            (0): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
            (1): Dropout(p=0.0, inplace=False)
            (2): Sequential(
              (0): TransposeLast()
              (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (2): TransposeLast()
            )
            (3): GELU(approximate='none')
          )
          (1-4): 4 x Sequential(
            (0): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
            (1): Dropout(p=0.0, inplace=False)
            (2): Sequential(
              (0): TransposeLast()
              (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (2): TransposeLast()
            )
            (3): GELU(approximate='none')
          )
          (5-6): 2 x Sequential(
            (0): Con

In [18]:
from main import *
#peft_model = peft_model.merge_and_unload()

model.ssl_model = W2V2_TA(import_fairseq_model(
    model.ssl_model.model
)).to(device)

ValueError: Unexpected key: encoder.layers.0.self_attn.k_proj.base_layer.weight

In [ ]:
config = LoraConfig(
    target_modules=["q_proj", "v_proj", "k_proj"],
)
peft_model = get_peft_model(model, config)




In [ ]:
peft_model

PeftModel(
  (base_model): LoraModel(
    (model): Distil_XLSR_N_Trans_Layer_VIB(
      (ssl_model): My_XLSR_FE(
        (model): Wav2Vec2Model(
          (feature_extractor): ConvFeatureExtractionModel(
            (conv_layers): ModuleList(
              (0): Sequential(
                (0): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
                (1): Dropout(p=0.0, inplace=False)
                (2): Sequential(
                  (0): TransposeLast()
                  (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
                  (2): TransposeLast()
                )
                (3): GELU(approximate='none')
              )
              (1-4): 4 x Sequential(
                (0): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
                (1): Dropout(p=0.0, inplace=False)
                (2): Sequential(
                  (0): TransposeLast()
                  (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
                  (2): Tr

In [ ]:
jit_model = torch.jit.trace(model, torch.randn(1, 16000).to(device), strict=False)

RuntimeError: 
Return value was annotated as having type str but is actually of type List[str]:
  File "/home/hungdx/miniconda3/envs/KDW2V2-AASISTL/lib/python3.10/site-packages/peft/tuners/tuners_utils.py", line 465
    def active_adapter(self) -> str:
        # use a property to ensure that active_adapter is not set directly, instead use the set_adapter method
        return self._active_adapter
        ~~~~~~~~~~~~~~~~~~~~~~~~~~~ <--- HERE


# Score fusion (Average)

In [ ]:
import sys
import os.path
import numpy as np
import pandas
import eval_metrics_DF as em
import os

score_file1 = '/datab/hungdx/KDW2V-AASISTL/W2V2BASE_Linear_DKDLoss_cnsl_noaudiomentations_best39_feb07_6.22.txt'
score_file2 = '/datab/hungdx/KDW2V-AASISTL/results/W2V2BASE_AASISTL_DKDLoss_cnsl_audiomentations_5_best11_feb07.txt'

submission_scores1 = pandas.read_csv(
    score_file1, sep=' ', header=None, skipinitialspace=True)
submission_scores2 = pandas.read_csv(
    score_file2, sep=' ', header=None, skipinitialspace=True)


# Calculate average score from two score files and save to a new file
average_score = (submission_scores1[1] + submission_scores2[1]) / 2
final_score = pandas.concat([submission_scores1[0], average_score], axis=1)

# Save final_score to a txt file
final_score.to_csv('W2V2BASE_Linear+W2V2BASE_AASISTL_fusion.txt',
                   sep=' ', header=False, index=False)

In [ ]:
import os
from pydub import AudioSegment

# Function to split the audio file and save segments to a specified folder
def split_audio(file_path, output_folder, segment_length_ms=4000, sample_rate=16000):
    # Ensure the output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Load the audio file
    audio = AudioSegment.from_wav(file_path)
    
    # Ensure the audio is at the correct frame rate
    if audio.frame_rate != sample_rate:
        audio = audio.set_frame_rate(sample_rate)
    
    # Length of the audio in milliseconds
    audio_length_ms = len(audio)
    
    # Split the audio into segments
    for i in range(0, audio_length_ms, segment_length_ms):
        # Calculate the end time of the segment
        end = i + segment_length_ms
        
        # Avoid going beyond the file length
        if end > audio_length_ms:
            end = audio_length_ms
        
        # Extract the segment
        segment = audio[i:end]
        
        # Create a filename for the segment
        segment_file_name = f"segment_{i//1000}-{end//1000}.wav"
        
        # Full path for the segment file
        segment_file_path = os.path.join(output_folder, segment_file_name)
        
        # Export the segment to a file
        segment.export(segment_file_path, format="wav")
        
        print(f"Exported {segment_file_path}")

# Example usage
split_audio("/datab/hungdx/KDW2V-AASISTL/U.S Presidents Play Minecraft 5 [TubeRipper.com].flac", "./folder")


CouldntDecodeError: Decoding failed. ffmpeg returned error code: 1

Output from ffmpeg/avlib:

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --ena

In [ ]:
# from wav2vec2_linear_nll_multi import Model as W2V2_NLL_Multi
# import torch


# device = "cuda" if torch.cuda.is_available() else "cpu"
# cp_path = "/datab/hungdx/KDW2V-AASISTL/xlsr2_300m.pt"
# model = W2V2_NLL_Multi(device, cp_path, out_dim=1024).to(device)
# checkpoint = "/datab/phucdt/SCL-Deepfake-audio-detection/model/pretrained/conf-5-linear-feb07-29-36-epoch46.pth"
# # Load the model
# model.load_state_dict(torch.load(checkpoint, map_location=device)) 


In [ ]:
# import librosa
# import torch


# input = torch.zeros(1, 16000).to(device)

# print(model(input))

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

# Load the audio file
# Update the path to your file
file = '/datab/hungdx/KDW2V-AASISTL/test_samples/screen/silence_screenrecording_trial_3/Silence_screenrecording_trial_3_17_FAKE 62.5%.wav'
y, sr = librosa.load(file, sr=16000)

# Function to compute and plot spectrogram


def plot_spectrogram(y, sr, title):
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    plt.figure(figsize=(14, 5))
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.show()


# Plot original spectrogram
plot_spectrogram(y, sr, 'Original Spectrogram')



# Trim silence
# You can adjust the top_db parameter as needed
# Assuming 'y' is your audio signal and 'sr' is the sample rate
# Set a threshold for silence (in dB)
silence_threshold = 20  # This is an example value; adjust based on your needs

y_trimmed, index = librosa.effects.trim(y, top_db=silence_threshold)
print(librosa.get_duration(y=y), librosa.get_duration(y=y_trimmed))

# Plot trimmed spectrogram with correct time axis
plot_spectrogram(y_trimmed, sr, 'Trimmed Spectrogram')

In [ ]:
# Plot original spectrogram
plot_spectrogram(y, sr, 'Original Spectrogram')

# Experiment with different top_db values
for silence_threshold in [_ for _ in range(0, 60, 1)]:
    y_trimmed, index = librosa.effects.trim(y, top_db=silence_threshold)
    duration_original = librosa.get_duration(y=y)
    duration_trimmed = librosa.get_duration(y=y_trimmed)
    print(f"Silence threshold: {silence_threshold} dB")
    print(f"Original duration: {duration_original:.2f} seconds")
    print(f"Trimmed duration: {duration_trimmed:.2f} seconds")
    plot_spectrogram(
        y_trimmed, sr, f'Trimmed Spectrogram (threshold {silence_threshold} dB)')